# 📝 Week 5: Text Embedding & Representasi Semantik

Notebook ini mengeksplorasi representasi teks sebagai vektor padat (dense vectors) menggunakan model Sentence-BERT untuk analisis semantik artikel MBG.

## 🎯 Tujuan Pembelajaran:
1. Memahami konsep text embedding dan perbedaannya dengan TF-IDF
2. Menggunakan Sentence-BERT untuk mengkodekan kalimat/dokumen ke vektor
3. Menghitung kemiripan semantik antar artikel menggunakan cosine similarity
4. Memvisualisasikan ruang embedding dengan PCA dan t-SNE
5. Melakukan clustering tematik dengan K-Means

## 📦 Instalasi Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

%pip install -q sentence-transformers scikit-learn matplotlib seaborn pandas openpyxl

print("✅ Semua library berhasil diinstall!")

## 📚 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Import libraries berhasil!")

---

## 1️⃣ Konsep Text Embedding

### 1.1 Kenapa Embedding?

| Representasi | Contoh | Kelemahan |
|---|---|---|
| Bag-of-Words | [1,0,1,0,1,...] | Sparser, tidak ada makna semantik |
| TF-IDF | [0.3,0,0.7,0,...] | Masih sparse, kata tidak saling berhubungan |
| **Word2Vec** | [0.12, -0.45, 0.87,...] | Kata per kata, tidak memahami kalimat |
| **Sentence-BERT** | [0.21, 0.34, -0.12,...] | **Dense, semantik-aware, level kalimat** |

### 1.2 Arsitektur Sentence-BERT

Sentence-BERT (Reimers & Gurevych, 2019) menggunakan **Siamese BERT** network:
```
Kalimat A → BERT Encoder → Pooling → Vektor A
Kalimat B → BERT Encoder → Pooling → Vektor B
                                → Cosine Similarity
```

---

## 2️⃣ Load Dataset

In [ ]:
# Load dataset MBG
file_path = 'artikel_manual_celine.xlsx'
df_main = pd.read_excel(file_path, sheet_name='Main')
df_case = pd.read_excel(file_path, sheet_name='Case')

print("✅ Dataset berhasil dimuat!")
print(f"  Main: {len(df_main)} artikel")
print(f"  Case: {len(df_case)} artikel")
df_main[['Judul', 'Kategori', 'Tone']].head(5)

In [ ]:
# Siapkan teks konten untuk embedding
sentences = df_main['Judul'].fillna('').tolist()
contents = df_main['Konten'].fillna('').tolist()
titles = df_main['Judul'].fillna('Article').tolist()
categories = df_main['Kategori'].fillna('Unknown').tolist()

print(f"\n✅ Siap mengkodekan {len(sentences)} artikel")
print(f"\nContoh judul:")
for i, s in enumerate(sentences[:3]):
    print(f"  [{i+1}] {s}")

---

## 3️⃣ Generate Sentence Embeddings

### 3.1 Load Model Sentence-BERT Multibahasa

In [ ]:
# Load model multilingual yang mendukung Bahasa Indonesia
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
print(f"⏳ Loading model '{model_name}'...")
model = SentenceTransformer(model_name)
print("✅ Model berhasil dimuat!")
print(f"  Max sequence length: {model.max_seq_length}")
print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")

### 3.2 Encode Judul Artikel

In [ ]:
# Encode semua judul artikel
print("⏳ Mengkodekan judul artikel...")
embeddings = model.encode(sentences, show_progress_bar=True)

print(f"\n✅ Embeddings selesai dibuat!")
print(f"  Shape: {embeddings.shape}")
print(f"  {embeddings.shape[0]} artikel × {embeddings.shape[1]} dimensi")

---

## 4️⃣ Cosine Similarity

Cosine similarity mengukur sudut antara dua vektor:
$$\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

Nilai berkisar 0 (berbeda total) hingga 1 (identik).

### 4.1 Hitung Similarity Matrix

In [ ]:
# Hitung cosine similarity antar semua dokumen
similarity_matrix = cosine_similarity(embeddings)

print("✅ Similarity matrix dihitung!")
print(f"  Shape: {similarity_matrix.shape}")
print(f"  Nilai min: {similarity_matrix.min():.4f}")
print(f"  Nilai max (diagonal): {similarity_matrix.max():.4f}")
print(f"  Rata-rata off-diagonal: {similarity_matrix[~np.eye(len(similarity_matrix), dtype=bool)].mean():.4f}")

### 4.2 Visualisasi Heatmap Similarity

In [ ]:
# Heatmap cosine similarity
n = min(20, len(sentences))  # tampilkan maksimal 20 artikel
short_titles = [t[:25] + '...' if len(t) > 25 else t for t in titles[:n]]

plt.figure(figsize=(14, 10))
sns.heatmap(
    similarity_matrix[:n, :n],
    xticklabels=short_titles,
    yticklabels=short_titles,
    cmap='YlOrRd',
    vmin=0, vmax=1,
    annot=True if n <= 10 else False,
    fmt='.2f' if n <= 10 else ''
)
plt.title('🔥 Heatmap Cosine Similarity Antar Artikel MBG', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('cosine_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Heatmap tersimpan sebagai 'cosine_similarity_heatmap.png'")

### 4.3 Cari Pasangan Artikel Paling Mirip

In [ ]:
# Cari top-5 pasangan artikel paling mirip (non-diagonal)
sim_pairs = []
for i in range(len(similarity_matrix)):
    for j in range(i + 1, len(similarity_matrix)):
        sim_pairs.append((i, j, similarity_matrix[i, j]))

sim_pairs.sort(key=lambda x: x[2], reverse=True)

print("📊 Top 5 pasangan artikel paling mirip secara semantik:")
print(f"{'Rank':<5} {'Sim Score':<12} {'Artikel A':<45} {'Artikel B':<45}")
print("-" * 110)
for rank, (i, j, score) in enumerate(sim_pairs[:5], 1):
    a = titles[i][:42] + '...' if len(titles[i]) > 42 else titles[i]
    b = titles[j][:42] + '...' if len(titles[j]) > 42 else titles[j]
    print(f"{rank:<5} {score:<12.4f} {a:<45} {b:<45}")

---

## 5️⃣ Visualisasi Ruang Embedding

Dimensi embedding (384) terlalu tinggi untuk divisualisasikan — kita reduksi ke 2D menggunakan **PCA** dan **t-SNE**.

### 5.1 Reduksi Dimensi dengan PCA

In [ ]:
# PCA 2D
pca = PCA(n_components=2, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)

print(f"✅ PCA selesai!")
print(f"  Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"  Total explained: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Plot PCA
fig, ax = plt.subplots(figsize=(12, 8))

unique_cats = list(set(categories))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_cats)))
cat_color = {cat: colors[i] for i, cat in enumerate(unique_cats)}

for cat in unique_cats:
    mask = [c == cat for c in categories]
    pts = embeddings_pca[mask]
    ax.scatter(pts[:, 0], pts[:, 1], label=cat, alpha=0.7, s=80)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
ax.set_title('📐 Visualisasi PCA Embedding Artikel MBG', fontsize=14)
ax.legend(title='Kategori', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('pca_embedding.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ PCA plot tersimpan sebagai 'pca_embedding.png'")

### 5.2 Reduksi Dimensi dengan t-SNE

In [ ]:
# t-SNE 2D
perplexity_val = min(30, len(embeddings) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity_val, n_iter=1000)
embeddings_tsne = tsne.fit_transform(embeddings)

print(f"✅ t-SNE selesai!")
print(f"  Perplexity: {perplexity_val}")

In [ ]:
# Plot t-SNE
fig, ax = plt.subplots(figsize=(12, 8))

for cat in unique_cats:
    mask = [c == cat for c in categories]
    pts = embeddings_tsne[np.array(mask)]
    ax.scatter(pts[:, 0], pts[:, 1], label=cat, alpha=0.7, s=80)

ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
ax.set_title('🔵 Visualisasi t-SNE Embedding Artikel MBG', fontsize=14)
ax.legend(title='Kategori', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('tsne_embedding.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ t-SNE plot tersimpan sebagai 'tsne_embedding.png'")

---

## 6️⃣ Clustering Tematik dengan K-Means

In [ ]:
# K-Means clustering (3 cluster)
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

# Label heuristik untuk cluster
cluster_names = {0: 'Challenge', 1: 'Solution', 2: 'Impact'}

df_main = df_main.copy()
df_main['cluster_id'] = cluster_labels
df_main['cluster_name'] = [cluster_names[l] for l in cluster_labels]

print("✅ K-Means clustering selesai!")
print("\n📊 Distribusi artikel per cluster:")
for cid, cname in cluster_names.items():
    count = (cluster_labels == cid).sum()
    print(f"  Cluster {cid} - {cname}: {count} artikel")

In [ ]:
# Visualisasi cluster pada t-SNE
cluster_colors = ['#2196F3', '#4CAF50', '#FF9800']
fig, ax = plt.subplots(figsize=(12, 8))

for cid, cname in cluster_names.items():
    mask = cluster_labels == cid
    pts = embeddings_tsne[mask]
    ax.scatter(pts[:, 0], pts[:, 1], label=f'Cluster {cid}: {cname}',
               color=cluster_colors[cid], alpha=0.75, s=100, edgecolors='white', linewidths=0.5)

ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
ax.set_title('🎯 K-Means Cluster pada Ruang Embedding Artikel MBG', fontsize=14)
ax.legend(title='Cluster Tematik', fontsize=10)
plt.tight_layout()
plt.savefig('kmeans_cluster.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Cluster plot tersimpan sebagai 'kmeans_cluster.png'")

In [ ]:
# Tampilkan sampel artikel per cluster
print("📋 Sampel artikel per cluster tema:")
for cid, cname in cluster_names.items():
    print(f"\n--- Cluster {cid}: {cname} ---")
    samples = df_main[df_main['cluster_id'] == cid]['Judul'].head(3).values
    for j, s in enumerate(samples, 1):
        print(f"  {j}. {s[:80]}")

---

## 📝 Kesimpulan

- 🎯 **Text Embedding** mengkonversi teks menjadi vektor padat yang mampu menangkap makna semantik — berbeda dengan TF-IDF yang hanya menghitung statistik kemunculan kata.
- 📊 **Cosine Similarity** memungkinkan pengukuran kemiripan semantik antar artikel secara kuantitatif; artikel yang membahas topik serupa memiliki nilai similarity tinggi.
- 🔧 **PCA dan t-SNE** memvisualisasikan ruang berdimensi tinggi (384D) menjadi 2D untuk analisis klaster dan pola distribusi tematik.
- 📂 **K-Means Clustering** mengelompokkan artikel ke dalam tema-tema laten (Challenge/Solution/Impact) yang mencerminkan framing berita MBG di media Indonesia.

---

## 📚 Referensi

- Mikolov, T., et al. (2013). Efficient estimation of word representations in vector space. *arXiv preprint arXiv:1301.3781*.
- Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. *EMNLP*.
- Van der Maaten, L., & Hinton, G. (2008). Visualizing data using t-SNE. *JMLR, 9*, 2579–2605.
- Sentence-Transformers docs: [https://www.sbert.net/](https://www.sbert.net/)

---

**Terima kasih! Happy Learning! 🎓**